# Sésame, ouvre-toi — Partie 2 : Feature Engineering & Modélisation

Ce notebook fait suite à `sesame_partie1_eda.ipynb` (récupération des 3 tables + analyse exploratoire comparative). Conformément au cahier des charges, il regroupe l'ensemble du **pipeline de modélisation** : feature engineering, entraînement, optimisation d'hyperparamètres, explicabilité (SHAP) et audit d'équité — construit **section par section**.

---
## Sommaire de ce notebook
- **2.1 Feature Engineering** *(cette section)*
- 2.2 Choix et entraînement du/des modèle(s)
- 2.3 Optimisation des hyperparamètres (Optuna)
- 2.4 Évaluation & métriques
- 2.5 Explicabilité (SHAP / LIME / SAGE)
- 2.6 Audit d'équité & stratégies de mitigation

---
# 2.1 Feature Engineering

**Objectif de cette section :** transformer la table brute *"cox-violent-parsed"* (granularité dossier/charge, avec colonnes administratives et données manquantes structurelles) en un **jeu de données d'apprentissage propre**, avec une cible binaire claire, des features justifiées une à une, et un split train/test qui préserve la représentativité des groupes sensibles — condition nécessaire pour que l'audit d'équité mené plus loin soit valide statistiquement.


In [1]:
# --- Imports ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

RANDOM_STATE = 42


## 2.1.1 Reconstruction du cohort "récidive à 2 ans"

La table `cox-violent-parsed` contient **une ligne par dossier judiciaire**, avec des cas où l'issue (récidive) n'est pas connue, des charges sans lien réel avec l'évaluation COMPAS (dossiers trop anciens/récents par rapport au screening) et des lignes dupliquées sans identifiant valide.

On applique ici la méthodologie de nettoyage documentée par ProPublica (*How We Analyzed the COMPAS Recidivism Algorithm*), en la justifiant étape par étape plutôt qu'en l'appliquant aveuglément :

| Filtre | Justification |
|---|---|
| `id` non manquant | Les lignes à `id` manquant sont des intervalles de suivi supplémentaires (format survie `start`/`end`/`event`) rattachés à un même individu déjà représenté par sa ligne principale. Les garder dupliquerait la même personne plusieurs fois et **biaiserait l'indépendance des observations**, une hypothèse nécessaire pour une classification standard. |
| `c_charge_degree` non manquant | Les charges sans degré (F/M) correspondent essentiellement à des infractions de type "ordonnance locale" (stationnement, trafic...) hors du périmètre pénal que COMPAS est censé évaluer. Les inclure ajouterait du **bruit non pertinent** à la tâche. |
| `days_b_screening_arrest` ∈ [-30, 30] | Ce délai mesure l'écart entre la date d'arrestation liée à la charge et la date de screening COMPAS. Un écart trop important signifie que le **score évalué n'est probablement pas celui de cette charge précise** (erreur d'appariement dans les données sources) — les conserver introduirait du bruit dans la relation feature → score. |
| `is_recid != -1` | La valeur -1 signifie que l'issue de récidive n'est **pas connue** (dossier trop récent pour observer 2 ans de suivi). Sans label fiable, ces lignes ne peuvent pas servir à l'apprentissage supervisé. |

Chaque filtre retire des lignes **pour une raison métier explicite**, ce qui diffère d'une suppression arbitraire de valeurs manquantes.


In [2]:
cox = pd.read_csv("data/coxviolentparsed.csv")
print(f"Table source (cox-violent-parsed) : {cox.shape[0]:,} lignes")

df = cox.copy()

etapes = []

df = df[df["id"].notna()]
etapes.append(("id non manquant", len(df)))

df = df[df["c_charge_degree"].notna()]
etapes.append(("c_charge_degree renseigné", len(df)))

df = df[(df["days_b_screening_arrest"] >= -30) & (df["days_b_screening_arrest"] <= 30)]
etapes.append(("days_b_screening_arrest dans [-30, 30]", len(df)))

df = df[df["is_recid"] != -1]
etapes.append(("is_recid connu (0 ou 1)", len(df)))

suivi = pd.DataFrame(etapes, columns=["Filtre appliqué", "Lignes restantes"])
suivi["Lignes retirées"] = suivi["Lignes restantes"].diff().fillna(cox.shape[0] - suivi["Lignes restantes"].iloc[0]).abs().astype(int)
display(suivi)

print(f"\nCohort final : {df.shape[0]:,} prévenus (contre {cox['id'].nunique():,.0f} avant filtrage qualité)")


Table source (cox-violent-parsed) : 18,316 lignes


,Filtre appliqué,Lignes restantes,Lignes retirées
0,id non manquant,11001,7315
1,c_charge_degree renseigné,10511,490
2,"days_b_screening_arrest dans [-30, 30]",8899,1612
3,is_recid connu (0 ou 1),8899,0



Cohort final : 8,899 prévenus (contre 11,001 avant filtrage qualité)


In [3]:
# Vérification : la structure démographique du cohort final reste-t-elle comparable à la table brute ?
comparaison = pd.DataFrame({
    "Table brute (id unique)": cox.drop_duplicates("id")["race"].value_counts(normalize=True).round(3),
    "Cohort final filtré": df["race"].value_counts(normalize=True).round(3),
}).fillna(0)
comparaison


,Table brute (id unique),Cohort final filtré
race,,
African-American,0.533,0.536
Caucasian,0.335,0.334
Hispanic,0.076,0.071
Other,0.048,0.053
Asian,0.004,0.004
Native American,0.003,0.002


**Lecture :** les proportions par groupe ethnique restent quasiment inchangées après filtrage (écarts < 1 point). Le nettoyage ne **déséquilibre pas artificiellement** la représentativité des groupes — condition importante puisque ce sont ces groupes qui seront comparés lors de l'audit d'équité.


## 2.1.2 Définition de la cible

La table dispose des colonnes `start`, `end`, `event`, qui permettent en théorie une **analyse de survie** (modèle de Cox — d'où le nom du fichier) : le risque de récidive pourrait être modélisé comme le temps écoulé avant l'événement plutôt que comme un simple binaire.

**Choix retenu : classification binaire sur `is_recid`** (a récidivé dans la fenêtre de suivi = 1/0), et non une analyse de survie complète.

*Justification (biais/variance/complexité) :*
- Le brief demande explicitement un **modèle de prédiction** comparable aux scores COMPAS eux-mêmes construits comme des scores de classification (`decile_score`, `score_text`) — une classification binaire est directement comparable et permet de réutiliser les métriques et l'appareillage d'explicabilité (SHAP) les plus matures pour ce type de sortie.
- Un modèle de survie ajoute un degré de complexité (fonction de risque, censure) sans bénéfice direct pour répondre à la question d'équité posée par le brief, qui compare des **groupes à un instant donné** (parité démographique, égalité des chances) — ce sont des métriques définies pour la classification.
- On documente ce choix comme **piste d'extension possible** plutôt que comme un impensé.


In [4]:
df["two_year_recid"] = df["is_recid"].astype(int)

print("Distribution de la cible :")
display(df["two_year_recid"].value_counts(normalize=True).rename("proportion").round(3))
print(f"\nRatio de déséquilibre : {df['two_year_recid'].value_counts().max() / df['two_year_recid'].value_counts().min():.2f}")


Distribution de la cible :


two_year_recid
0    0.514
1    0.486
Name: proportion, dtype: float64


Ratio de déséquilibre : 1.06


**Lecture :** la cible est quasiment équilibrée (~52 %/48 %). Un déséquilibre aussi faible **ne justifie pas** de sur/sous-échantillonnage (SMOTE, class_weight agressif) au stade du modèle principal — cela reviendrait à ajouter de la variance artificielle sans bénéfice, la métrique d'accuracy ne serait déjà pas trompeuse ici. On gardera toutefois `class_weight="balanced"` en option testable lors de l'optimisation d'hyperparamètres (section 2.3), et le rappel/precision par classe comme métriques principales plutôt que la seule accuracy (section 2.4).


## 2.1.3 Sélection, transformation et encodage des features

### Variable à traiter avec précaution : le score COMPAS lui-même

`decile_score` et `score_text` sont **exclus du vecteur d'entrée `X`** du modèle principal.

*Justification :* l'objectif du projet est d'auditer COMPAS, pas de le reproduire. Inclure le score COMPAS comme feature créerait une **fuite de cible circulaire** : COMPAS a été calibré pour prédire `is_recid`, donc l'utiliser comme predictor donnerait un modèle qui ne fait qu'imiter (et donc hériter mécaniquement) le biais qu'on cherche à mesurer, rendant l'audit tautologique. On conserve ces colonnes **à part** (non utilisées pour l'entraînement) afin de comparer, en section 2.4, les performances et l'équité de *notre* modèle indépendant vs. celles du score COMPAS existant.

### Autres features retenues


In [5]:
# --- Age : on garde la variable numérique brute plutôt que la version discrétisée `age_cat` ---
# Justification : la discrétisation en 3 classes ("< 25", "25-45", "> 45") fait perdre de l'information
# ordinale sans bénéfice de robustesse ici (peu de valeurs aberrantes sur l'âge) -> variance inutile.
df["age"] = df["age"].astype(int)

# --- Sexe : encodage binaire simple (2 catégories, pas d'ordre) ---
df["is_male"] = (df["sex"] == "Male").astype(int)

# --- Antécédents judiciaires ---
# Justification : priors_count reste séparé (variable continue à forte valeur prédictive, cf. partie 1).
# Les 3 compteurs de délits juvéniles (juv_fel_count / juv_misd_count / juv_other_count) sont
# à >90% égaux à 0 individuellement (forte parcimonie) -> on les agrège en un compteur total
# `juv_total` pour réduire la dimensionnalité et la variance d'estimation sur des variables trop
# rares prises séparément, tout en conservant le signal "a un passé judiciaire de mineur".
df["juv_total"] = df["juv_fel_count"] + df["juv_misd_count"] + df["juv_other_count"]

# --- Gravité de la charge ---
# Justification : c_charge_degree contient 14 modalités très déséquilibrées (cf. partie 1).
# On le simplifie en un binaire Félonie/Délit (is_felony), qui capture la distinction pénale
# la plus significative sans exploser la dimensionnalité avec du one-hot sur 14 catégories
# rares (risque de surapprentissage sur des modalités à très faible effectif).
df["is_felony"] = df["c_charge_degree"].str.contains("F", na=False).astype(int)

features_num_bin = ["age", "is_male", "priors_count", "juv_total", "is_felony"]
df[features_num_bin].describe().T


,count,mean,std,min,25%,50%,75%,max
age,8899.0,33.865378,11.535246,18.0,25.0,31.0,41.0,96.0
is_male,8899.0,0.821441,0.383004,0.0,1.0,1.0,1.0,1.0
priors_count,8899.0,3.870997,5.229976,0.0,0.0,2.0,5.0,37.0
juv_total,8899.0,0.299921,1.036451,0.0,0.0,0.0,0.0,21.0
is_felony,8899.0,0.698505,0.458933,0.0,0.0,1.0,1.0,1.0


### Variable sensible : `race`

`race` n'est **pas incluse** dans le jeu de features principal `X` (approche dite *"fairness through unawareness"*), afin de tester une question centrale de l'audit : **retirer l'attribut sensible suffit-il à supprimer le biais ?**

On construit donc **deux jeux de features en parallèle** :
- `X` (sans `race`) → modèle principal, audité *a posteriori* par groupe.
- `X_avec_race` (avec `race` one-hot encodée) → modèle de comparaison, pour objectiver si la performance/l'équité change significativement lorsque l'attribut est explicitement disponible.

*Hypothèse à vérifier plus loin (section 2.6) :* la littérature (et l'analyse de la partie 1, où `priors_count` était déjà corrélé à `race`) suggère que retirer `race` ne suffit généralement pas, car d'autres variables (`priors_count` notamment) agissent comme **proxy** de l'attribut sensible.


In [6]:
# On conserve la race non pas comme feature exploitée pour la prédiction (sauf variante de comparaison),
# mais comme attribut d'audit "à part" (nécessaire pour calculer les métriques de fairness en 2.6).
races_principales = ["African-American", "Caucasian"]
# Justification : les groupes Hispanic/Other/Asian/Native American représentent chacun <10% de
# l'échantillon (cf. partie 1) -> les métriques de fairness calculées dessus auraient une variance
# d'estimation trop grande pour être interprétées de façon fiable. On les conserve dans le dataset
# complet, mais les comparaisons d'équité se concentreront sur les 2 groupes majoritaires.
df["race_agg"] = np.where(df["race"].isin(races_principales), df["race"], "Other/Minorité")

print(df["race_agg"].value_counts())


race_agg
African-American    4767
Caucasian           2974
Other/Minorité      1158
Name: count, dtype: int64


In [7]:
race_dummies = pd.get_dummies(df["race_agg"], prefix="race", drop_first=True)  # drop_first : évite la colinéarité parfaite (piège de la variable dummy)
race_dummies.head()


,race_Caucasian,race_Other/Minorité
0,False,True
1,False,True
3,False,False
4,False,False
5,False,False


## 2.1.4 Construction des jeux de features finaux

Récapitulatif des colonnes retenues :

| Variable | Type | Traitement | Dans `X` principal ? |
|---|---|---|---|
| `age` | numérique | brute | ✅ |
| `is_male` | binaire | encodage 0/1 | ✅ |
| `priors_count` | numérique | brute | ✅ |
| `juv_total` | numérique | somme des 3 compteurs juvéniles | ✅ |
| `is_felony` | binaire | simplification de `c_charge_degree` | ✅ |
| `race` | catégorielle | one-hot (agrégée) | ❌ (uniquement dans `X_avec_race`) |
| `decile_score` / `score_text` | — | conservées à part | ❌ (fuite de cible) |
| `two_year_recid` | binaire | — | Cible `y` |


In [8]:
FEATURES_BASE = ["age", "is_male", "priors_count", "juv_total", "is_felony"]

X = df[FEATURES_BASE].copy()
X_avec_race = pd.concat([X, race_dummies], axis=1)
y = df["two_year_recid"].copy()

# Attributs conservés à part pour l'audit d'équité et la comparaison avec COMPAS (non utilisés pour l'apprentissage)
audit_attrs = df[["race", "race_agg", "decile_score", "score_text", "sex"]].copy()

print("X (sans race)       :", X.shape)
print("X_avec_race         :", X_avec_race.shape)
print("y (cible)            :", y.shape)
X.head()


X (sans race)       : (8899, 5)
X_avec_race         : (8899, 7)
y (cible)            : (8899,)


,age,is_male,priors_count,juv_total,is_felony
0,69,1,0,0,1
1,69,1,0,0,1
3,34,1,0,0,1
4,24,1,4,1,1
5,24,1,4,1,1


## 2.1.5 Split train/test stratifié

**Choix : stratification conjointe sur la cible `y` ET sur le groupe `race_agg`.**

*Justification :* une stratification sur `y` seule garantirait un ratio de classes identique entre train et test, mais **pas** une représentativité comparable des groupes ethniques dans le jeu de test. Or l'audit d'équité de la section 2.6 sera calculé **sur le test set** : si un groupe minoritaire s'y retrouve sous-représenté par hasard, les métriques de fairness (calculées par sous-groupe) auraient une variance d'estimation excessive, potentiellement trompeuse. Stratifier sur les deux variables conjointement limite ce risque.


In [9]:
strate = df["two_year_recid"].astype(str) + "_" + df["race_agg"]

X_train, X_test, y_train, y_test, audit_train, audit_test = train_test_split(
    X, y, audit_attrs,
    test_size=0.2,
    stratify=strate,
    random_state=RANDOM_STATE,
)

# On aligne le split "avec race" sur les mêmes index (mêmes individus dans train/test que ci-dessus)
X_avec_race_train = X_avec_race.loc[X_train.index]
X_avec_race_test = X_avec_race.loc[X_test.index]

print(f"Train : {X_train.shape[0]:,} individus  |  Test : {X_test.shape[0]:,} individus")
print("\nVérification de la stratification (proportion de la cible) :")
print(pd.DataFrame({"train": y_train.value_counts(normalize=True), "test": y_test.value_counts(normalize=True)}).round(3))

print("\nVérification de la représentativité des groupes ethniques :")
print(pd.DataFrame({
    "train": audit_train["race_agg"].value_counts(normalize=True),
    "test": audit_test["race_agg"].value_counts(normalize=True),
}).round(3))


Train : 7,119 individus  |  Test : 1,780 individus

Vérification de la stratification (proportion de la cible) :
                train   test
two_year_recid              
0               0.514  0.514
1               0.486  0.486

Vérification de la représentativité des groupes ethniques :
                  train   test
race_agg                      
African-American  0.536  0.535
Caucasian         0.334  0.335
Other/Minorité    0.130  0.130


**Lecture :** les proportions de la cible et des groupes ethniques sont quasi identiques entre train et test (écarts < 1 point) — le split est donc représentatif sur les deux dimensions qui comptent pour la suite du projet (performance ET équité).


## 2.1.6 Sauvegarde des jeux de données

On sauvegarde des fichiers CSV intermédiaires afin que la section suivante (2.2 — choix et entraînement du modèle) puisse reprendre directement ces jeux de données propres, sans dépendre de l'exécution préalable de cette section dans la même session.


In [10]:
import os
os.makedirs("data/processed", exist_ok=True)

X_train.assign(split="train").to_csv("data/processed/X_train.csv", index=True)
X_test.assign(split="test").to_csv("data/processed/X_test.csv", index=True)
X_avec_race_train.to_csv("data/processed/X_avec_race_train.csv", index=True)
X_avec_race_test.to_csv("data/processed/X_avec_race_test.csv", index=True)
y_train.to_csv("data/processed/y_train.csv", index=True)
y_test.to_csv("data/processed/y_test.csv", index=True)
audit_train.to_csv("data/processed/audit_train.csv", index=True)
audit_test.to_csv("data/processed/audit_test.csv", index=True)

print("Fichiers sauvegardés dans data/processed/ :")
for f in sorted(os.listdir("data/processed")):
    print(" -", f)


Fichiers sauvegardés dans data/processed/ :
 - X_avec_race_test.csv
 - X_avec_race_train.csv
 - X_test.csv
 - X_train.csv
 - audit_test.csv
 - audit_train.csv
 - y_test.csv
 - y_train.csv


## 2.1.7 Synthèse de la section Feature Engineering

**Décisions prises et justifications :**

1. **Cohort reconstruit** (11 001 → 8 899 individus) via un filtrage qualité justifié métier (appariement score/charge, issue connue), sans déséquilibrer la répartition par groupe ethnique.
2. **Cible** : classification binaire `two_year_recid` — choix pragmatique par rapport à une analyse de survie, documenté comme piste d'extension.
3. **Features** réduites et justifiées une à une (agrégation des compteurs juvéniles rares, simplification de la gravité de charge, exclusion du score COMPAS pour éviter la fuite de cible).
4. **`race` volontairement exclue** du jeu de features principal (fairness through unawareness), avec un jeu alternatif `X_avec_race` préparé pour tester si cette exclusion suffit réellement à limiter le biais — question qui sera tranchée quantitativement en section 2.6.
5. **Split stratifié** sur cible + groupe ethnique pour garantir la validité statistique de l'audit d'équité mené sur le test set.

➡️ **Prochaine étape (2.2) :** choix justifié d'un ou plusieurs modèles de Machine Learning adaptés à la tâche, entraînement baseline, puis optimisation des hyperparamètres (Optuna).
